### Peat forest disturbance drivers in the Congo peatlands, 2022 onward
*Analysis design after Nesha et al. (2024), Environ. Res. Lett. 19 104031, [doi:10.1088/1748-9326/ad6679](https://doi.org/10.1088/1748-9326/ad6679), using the WUR driver-attributed alerts.*

| Step | What |
|---|---|
| Disturbances | WUR driver maps only (integrated RADD + GLAD-L + GLAD-S2 alerts, + DIST from 2026): historical 2022–2025 + operational 2026 → |
| Study area | Land cover: **palm-dominated peat (4)**, **hardwood-dominated peat (5)**, **buffer (6)**, restricted to the selected country |
| Context | Protected areas, forest concessions, admin units (per polygon) |
| Analyses | Drivers and area by class · protected areas, concessions, admin units · event sizes · monthly patterns · 1 km hotspots · distance to peat edge, roads and rivers |

**How to run it (users).** Set `ASSET_ROOT` in §1 to a folder in your own Earth Engine project, then Run All. The notebook reads the shared stack (`STACK_ASSET`), exports stats, events and the 1 km grid into your folder (8b), waits for them, and makes all tables, graphs and maps (saved to `outputs_peat_disturbance/`). You need read access to the shared stack and the input layers listed in §3.

**Quarterly update (stack owner).** Set `OVERWRITE = True` and Run All: 8a rebuilds the stack at `STACK_ASSET` (≈1–2 h; it is unavailable to users meanwhile) and 8b then rebuilds the outputs. Set it back to `False` afterwards. A stack that does not exist yet is always built. Anyone running the notebook afterwards gets fresh outputs automatically: outputs older than the stack are re-exported, and downloaded tables older than their asset are downloaded again.

#### 0 · Setup

In [ ]:
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display
import ee

EE_PROJECT = "fao-sepal-361210"   # processing project (SEPAL)
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine API", ee.__version__, "| project:", EE_PROJECT)

#### 1 · Configuration

In [ ]:
# ── Where things are ────────────────────────────────────────────────────────
STACK_ASSET = "projects/congopeatlands/assets/stack_wur_dem_2022_2026"   # shared stack (used if it exists, built here if not)
ASSET_ROOT = "projects/congopeatlands/assets"           # YOUR folder for stats, events and grid (write access needed)
LOCAL_OUT = Path("outputs_peat_disturbance"); LOCAL_OUT.mkdir(exist_ok=True)
OVERWRITE = False                                       # True = rebuild everything: the stack (if you can write to it) and your outputs
REFRESH_TABLES = False                                  # True = download the tables again even if the local copy is current

# ── Period and country (must match how the stack was built) ─────────────────
START_YEAR = 2022                                       # WUR drivers start Jan 2022
END_YEAR = pd.Timestamp.now().year                      # current year (partial)
COUNTRY_LIST = ["Democratic Republic of the Congo"]     # FAO GAUL ADM0_NAME; add "Congo" for ROC

# ── Study area: land-cover classes ──────────────────────────────────────────
LANDCOVER_ASSET = "projects/congopeatlands/assets/LandCover_Crezee_2022_500mbuffer"
LANDCOVER_CLASSES = {4: "palm-dominated peat", 5: "hardwood-dominated peat", 6: "buffer"}
PEAT_CLASS_CODES = [4, 5]                               # peat (distance-to-edge analysis uses these)
ZONE_COLORS = {"palm-dominated peat": "#66c2a4", "hardwood-dominated peat": "#006d2c", "buffer": "#fdd49e"}

# ── Drivers (WUR) ───────────────────────────────────────────────────────────
DRIVERS_HIST_ASSET = "projects/wurnrt-drivers/assets/distribution/driverclassification_afr_202201_202512"
DRIVERS_OPER_ASSET = "projects/wurnrt-drivers/assets/operational_postprocessed_afr_collection"
DRIVER_BAND, DRIVER_DATE_BAND = "class", "date"         # date = YYdoy
DRIVER_CLASSES = {
    1: "Small-scale agriculture", 2: "Small-scale agriculture (fire)",
    3: "Large-scale agriculture", 4: "Large-scale agriculture (fire)",
    5: "Road development", 6: "Selective logging", 7: "Mining", 8: "Flooding",
    9: "Wildfire", 10: "Other natural disturbance", 11: "Uncertain classification"}
FIRE_SUBTYPES = {2: 1, 4: 3}                            # 'with fire' subtype → parent class (grouped in graphs)
DRIVER_GROUP_COLORS = {
    "Small-scale agriculture": "#e6ab02", "Large-scale agriculture": "#a6761d", "Road development": "#525252",
    "Selective logging": "#1b9e77", "Mining": "#7570b3", "Flooding": "#1f78b4", "Wildfire": "#d95f02",
    "Other natural disturbance": "#66a61e", "Uncertain classification": "#bdbdbd"}

# ── Distances ───────────────────────────────────────────────────────────────
PEAT_DIST_ASSET = "projects/congopeatlands/assets/Crezee_peat_distance"
PEAT_EDGE_BAND = "dist_to_edge_m"                       # inside peat: distance to the peat edge (m)
ROADS_ASSET = "projects/cafi_fao_congo/regional/roads_Kleinschroth_etal_2019"
RIVERS_LINES = "projects/congopeatlands/assets/COD_waterways_lines"        # HOTOSM
WATER_POLY = "projects/congopeatlands/assets/COD_waterways_poly"
RIVER_TYPE_FIELD, RIVER_TYPES = "waterway", None        # e.g. ["river", "canal"]; None = all lines
ACCESS_SCALE, ACCESS_NEIGH_PX = 100, 300                # road/river distance at 100 m, search radius 30 km
EDGE_BINS_KM = [0.5, 1, 2, 3, 4, 5, 10]
ACCESS_BINS_KM = [1, 2, 3, 4, 5, 10, 15, 20, 25]

# ── Protected areas, concessions, admin units ──────────────────────────────
PA_ASSET = "projects/congopeatlands/assets/peatlands_PAs"
CONCESSIONS_ASSET = "projects/congopeatlands/assets/forest_concessions_COD"
ADMIN_ASSET = "projects/congopeatlands/assets/peatlands_adm1"
PA_NAME_FIELD = CONC_NAME_FIELD = None     # property with the polygon name (None = auto-detect)
ADMIN_NAME_FIELD = "gaul1_name"            # admin units reported by this property (e.g. "gaul1_name"; None = auto-detect)

# ── Analysis parameters (as in Nesha et al. 2024) ───────────────────────────
SIZE_CLASSES_HA = [0.5, 2]                              # small < 0.5 ≤ medium < 2 ≤ large
INTENSITY_THRESHOLDS_PCT = [2, 20]                      # 1 km cell: low < 2 % ≤ medium ≤ 20 % < high (hotspot)
TEMPORAL_HOTSPOT_PCT = 10                               # month with > 10 % of the year's disturbance
MMU_HA = 0.1                                            # smaller events are flagged (intersection fragments)

# ── Compute and maps ────────────────────────────────────────────────────────
TILE_DEG = 0.5                                          # processing tiles (only tiles with study area are used)
TILE_SCALE = 8
WAIT_POLL_MIN, WAIT_MAX_H = 5, 12                       # 8b waits for running exports: check every 5 min, give up after 12 h
TEST_BOX = [18.4, 2.8, 18.9, 3.3]                       # dry-run box — move it if it has no alerts
MAP_RES_DEG = 0.01                                      # finest static-map resolution (~1 km)
MAP_BASEMAP = "Esri.WorldTopoMap"                       # interactive preview: "Esri.WorldTopoMap" | "Esri.WorldImagery"
DOM_CELL_DEG = 0.1                                      # cell size of the dominant-driver map
GRID_DEG = 0.009                                        # hotspot grid cell ≈ 1 km at the equator

# ── Local exports for GIS and dashboards (§17) ──────────────────────────────
EXPORT_GEOTIFF = True                                   # ~1 km multi-band GeoTIFF of the grid maps
EXPORT_DASHBOARD_TABLES = True                          # tidy CSVs with coordinates / join keys

# ── Derived ─────────────────────────────────────────────────────────────────
ASSET_ROOT = ASSET_ROOT.rstrip("/")
START_YEAR = max(START_YEAR, 2022)
YEARS = list(range(START_YEAR, END_YEAR + 1)); N_YEARS = len(YEARS)
ZONE_CODES = list(LANDCOVER_CLASSES)                    # zone index = position + 1
ZONE_ORDER = list(LANDCOVER_CLASSES.values())
RUN_TAG = STACK_ASSET.rstrip("/").split("/")[-1].replace("stack_", "", 1)   # output names follow the stack name
print(f"Stack: {STACK_ASSET}\nOutputs: {ASSET_ROOT}/{{stats,events,grid1km_tbl}}_{RUN_TAG}")

#### 2 · Helpers

In [ ]:
def asset_type(path):
    """'IMAGE', 'IMAGE_COLLECTION', 'TABLE', 'FOLDER' or None (missing / no access)."""
    if not path:
        return None
    path = path.rstrip("/")
    try:
        if path.endswith("/assets"):
            ee.data.listAssets({"parent": path, "pageSize": 1}); return "FOLDER"
        return ee.data.getAsset(path)["type"]
    except (ee.EEException, TypeError):
        return None

def asset_time(path):
    """Last update time of an asset (UTC) or None."""
    try:
        return pd.Timestamp(ee.data.getAsset(path.rstrip("/"))["updateTime"])
    except Exception:
        return None

def geom_rings(g):
    """All outer rings / lines of a GeoJSON geometry as arrays (Polygon, MultiPolygon, GeometryCollection, lines)."""
    t, c = (g or {}).get("type"), (g or {}).get("coordinates")
    if t == "Polygon": return [np.array(c[0])]
    if t == "MultiPolygon": return [np.array(p[0]) for p in c]
    if t == "LineString": return [np.array(c)]
    if t == "MultiLineString": return [np.array(l) for l in c]
    if t == "GeometryCollection": return [r for sub in g.get("geometries", []) for r in geom_rings(sub)]
    return []

def ensure_folder(path):
    if path.rstrip("/").endswith("/assets"):
        return
    if asset_type(path) is None:
        ee.data.createAsset({"type": "FOLDER"}, path); print("created folder", path)

def load_image(asset, band=None):
    t = asset_type(asset)
    if t not in ("IMAGE", "IMAGE_COLLECTION"):
        return None
    img = ee.ImageCollection(asset).mosaic() if t == "IMAGE_COLLECTION" else ee.Image(asset)
    return img.select(band) if band else img.select(0)

def yydoy_to_ts(v):
    v = int(v)
    return pd.Timestamp(f"{2000 + v // 1000}-01-01") + pd.Timedelta(days=v % 1000 - 1)

def doy_to_month(year_img, doy_img):
    leap = year_img.mod(4).eq(0)
    doy = doy_img.subtract(leap.And(doy_img.gt(59)))
    month = ee.Image(1)
    for cum in [31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]:
        month = month.add(doy.gt(cum))
    return month.int()

def bin_index(img, edges):
    b = ee.Image(0)
    for e in edges:
        b = b.add(img.gt(e))
    return b.int()

def cum_labels(edges):
    return [f"≤{e:g} km" for e in edges] + ["all"]

def distance_km(source, scale, neigh_px):
    """Euclidean distance (km) to the nearest non-zero pixel of `source`, at `scale` m; beyond the radius = 'far'."""
    far = neigh_px * scale / 1000.0
    d = (source.unmask(0).gt(0)
         .fastDistanceTransform(neigh_px, "pixels", "squared_euclidean")
         .sqrt().multiply(ee.Image.pixelArea().sqrt()).divide(1000)
         .reproject(crs="EPSG:4326", scale=scale))
    return d.unmask(far)

def encode(imgs, fields):
    code = ee.Image(0).int()
    for name, size in fields:
        code = code.multiply(size).add(ee.Image(imgs[name]).int())
    return code.int()

def decode(codes, fields):
    rem = pd.Series(codes).round().astype("int64").to_numpy().copy()
    out = {}
    for name, size in reversed(fields):
        out[name] = rem % size
        rem //= size
    return pd.DataFrame(out)[[n for n, _ in fields]]

TASKS = {}
def active_tasks():
    """{description: (state, id)} of this account's READY / RUNNING Earth Engine tasks (also from earlier sessions)."""
    out = {}
    try:
        for op in ee.data.listOperations():
            md = op.get("metadata", {})
            if md.get("state") in ("PENDING", "RUNNING", "CANCELLING"):
                out[md.get("description")] = (md.get("state"), op.get("name", "").split("/")[-1])
    except Exception as e:
        print("⚠ could not list running tasks:", e)
    return out

def start_export(name, make_task, asset_id, overwrite=None, newer_than=None):
    """Start an export unless the asset exists and is up to date (or the same task is already running)."""
    overwrite = OVERWRITE if overwrite is None else overwrite
    short = asset_id.split("/assets/")[-1]
    if asset_type(asset_id) is not None:
        stale = newer_than is not None and (asset_time(newer_than) or pd.Timestamp.min.tz_localize("UTC")) > asset_time(asset_id)
        if not (overwrite or stale):
            print(f"[skip] {short} exists and is up to date"); return None
        print(f"[rebuild] {short} — {'stack is newer' if stale else 'overwrite requested'}")
        ee.data.deleteAsset(asset_id)
    task = make_task()
    desc = task.config.get("description") if hasattr(task, "config") else None
    running = active_tasks()
    if desc in running:
        print(f"[skip] {desc} is already {running[desc][0]} ({running[desc][1]}) — not starting a duplicate"); return None
    task.start(); TASKS[name] = task
    print(f"[started] {name}: {task.id}"); return task

def fc_to_df(fc):
    return ee.data.computeFeatures({"expression": ee.FeatureCollection(fc), "fileFormat": "PANDAS_DATAFRAME"})

def table_df(asset, selectors, refresh=None):
    """Stored EE table → DataFrame via one CSV download, cached in LOCAL_OUT/ee_tables (re-downloaded if the asset is newer)."""
    import urllib.request
    cache = LOCAL_OUT / "ee_tables"; cache.mkdir(exist_ok=True)       # raw downloads, kept apart from result CSVs
    path = cache / f"{asset.split('/')[-1]}.csv"
    if refresh is None:
        refresh = REFRESH_TABLES
    if path.exists() and not refresh:                 # newer asset than the local copy → download again
        t = asset_time(asset)
        refresh = t is not None and t > pd.Timestamp(path.stat().st_mtime, unit="s", tz="UTC")
    if refresh or not path.exists():
        n = ee.FeatureCollection(asset).size().getInfo()
        print(f"downloading {asset.split('/')[-1]} ({n:,} rows) …", end=" ", flush=True)
        url = ee.FeatureCollection(asset).getDownloadURL(filetype="csv", selectors=selectors,
                                                         filename=path.stem)
        urllib.request.urlretrieve(url, path)
        print("done")
    df = pd.read_csv(path, low_memory=False)
    for c in df.columns:                              # numbers stored as text (e.g. empty / null values) → numeric
        if c in (".geo", "kind") or pd.api.types.is_numeric_dtype(df[c]):
            continue
        num = pd.to_numeric(df[c], errors="coerce")
        bad = num.isna() & df[c].notna()
        if bad.any():
            print(f"⚠ {path.stem}.{c}: {bad.sum():,} non-numeric values set to missing, e.g. {df.loc[bad, c].unique()[:5].tolist()}")
        df[c] = num
    if ".geo" in df:
        xy = df.pop(".geo").map(lambda s: json.loads(s)["coordinates"])
        df["lon"], df["lat"] = xy.str[0], xy.str[1]
    return df

def pct(s):
    return s / s.sum() * 100

def cum_share(df, col, edges, weight=None):
    """Cumulative % by distance bin; weight=None counts rows (events), else sums that column."""
    n = len(edges) + 1
    s = (df.groupby(col)[weight].sum() if weight else df.groupby(col).size()).reindex(range(n), fill_value=0)
    return pd.Series((s.cumsum() / max(s.sum(), 1e-9) * 100).to_numpy(), index=cum_labels(edges))

NAME_CANDIDATES = ["NAME", "Name", "name", "NOM", "Nom", "nom", "ORIG_NAME", "gaul1_name", "GAUL1_NAME", "ADM1_NAME",
                   "ADM1_FR", "ADM1_EN", "NAME_1", "shapeName", "DESIG", "concession", "Concession", "titulaire", "TITULAIRE"]
def name_field(fc, preferred=None):
    props = [p for p in ee.Feature(ee.FeatureCollection(fc).first()).propertyNames().getInfo() if not p.startswith("system:")]
    if preferred:
        if preferred not in props:
            raise ValueError(f"{preferred!r} not found; available: {props}")
        return preferred
    return next((c for c in NAME_CANDIDATES if c in props), props[0] if props else None)

from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter
def plain_log(axis):
    """Label a log axis with plain numbers (0.1, 1, 10, 1,000) instead of powers of ten."""
    axis.set_major_locator(LogLocator(base=10, subs=(1.0, 2.0, 5.0)))
    axis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}" if v >= 1 else f"{v:g}"))
    axis.set_minor_formatter(NullFormatter())

FIG_DIR = LOCAL_OUT / "figures"; FIG_DIR.mkdir(exist_ok=True)
def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}_{RUN_TAG}.png", dpi=200, bbox_inches="tight")

#### 3 · Check assets and driver maps

In [ ]:
ASSETS = {"Shared stack": STACK_ASSET, "Land cover (required)": LANDCOVER_ASSET, "Peat distance": PEAT_DIST_ASSET,
          "Drivers, historical": DRIVERS_HIST_ASSET, "Drivers, operational": DRIVERS_OPER_ASSET,
          "Roads": ROADS_ASSET, "River lines": RIVERS_LINES, "Water polygons": WATER_POLY,
          "Protected areas": PA_ASSET, "Forest concessions": CONCESSIONS_ASSET, "Admin units": ADMIN_ASSET,
          "Output folder": ASSET_ROOT}
chk = pd.DataFrame([{"layer": k, "asset": a.split("/assets/")[-1], "type": asset_type(a) or "✗ not readable"}
                    for k, a in ASSETS.items()]).set_index("layer")
display(chk)
if chk.loc["Land cover (required)", "type"].startswith("✗"):
    raise RuntimeError("The land-cover layer is required — fix the path in §1 or ask for read access")
STACK_EXISTS = chk.loc["Shared stack", "type"] == "IMAGE"
BUILD_STACK = OVERWRITE or not STACK_EXISTS             # rebuild on request, build if missing (8a)
if not STACK_EXISTS:
    print(f"⚠ No readable stack at {STACK_ASSET} — it will be BUILT there in 8a (≈1–2 h, needs write access).\n"
          f"  If you meant to use an existing stack, check the path or ask its owner for read access.")
elif OVERWRITE:
    print(f"↻ OVERWRITE = True — the stack {STACK_ASSET} will be REBUILT in 8a (if you have write access)")
else:
    print(f"✓ Using the existing stack {STACK_ASSET}")
if BUILD_STACK and chk.loc["Peat distance", "type"].startswith("✗"):
    raise RuntimeError("The peat distance layer is required to build the stack")

# driver collections → one collection with 'class' and 'date'
dcol, rows = None, []
for name, asset in [("historical", DRIVERS_HIST_ASSET), ("operational", DRIVERS_OPER_ASSET)]:
    t = asset_type(asset)
    col = (ee.ImageCollection(asset) if t == "IMAGE_COLLECTION" else
           ee.ImageCollection([ee.Image(asset)]) if t == "IMAGE" else None)
    row = {"collection": name, "type": t or "✗ not readable"}
    if col is not None:
        info = ee.Dictionary({"n": col.size(), "bands": col.first().bandNames(),
                              "t0": col.aggregate_min("system:time_start"),
                              "t1": col.aggregate_max("system:time_start")}).getInfo()
        to_d = lambda ms: pd.to_datetime(ms, unit="ms").date() if ms is not None else "–"
        row.update({"images": info["n"], "bands": ", ".join(info["bands"]),
                    "first image": to_d(info["t0"]), "last image": to_d(info["t1"])})
        col = col.map(lambda i: i.select([DRIVER_BAND, DRIVER_DATE_BAND]).toInt32())
        dcol = col if dcol is None else dcol.merge(col)
    rows.append(row)
display(pd.DataFrame(rows).set_index("collection").fillna("–"))
if dcol is None and BUILD_STACK:
    raise RuntimeError("Neither driver collection is readable — needed to build the stack")

### Part A · Build layers in Earth Engine

#### 4 · Study area: land-cover classes, country, processing tiles
Each pixel gets a zone index: 1 = palm-dominated peat (class 4), 2 = hardwood-dominated peat (5), 3 = buffer (6). Everything is masked to the selected country, and only the tiles that contain study area are processed.

In [ ]:
gaul0 = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.inList("ADM0_NAME", COUNTRY_LIST))
found = gaul0.aggregate_array("ADM0_NAME").distinct().getInfo()
assert set(found) == set(COUNTRY_LIST), f"Not in GAUL: {set(COUNTRY_LIST) - set(found)}"
in_country = ee.Image(0).byte().paint(gaul0, 1)

lc = load_image(LANDCOVER_ASSET)
LC_PROJ = lc.projection()
zone = (lc.remap(ZONE_CODES, list(range(1, len(ZONE_CODES) + 1)), 0).unmask(0)
        .multiply(in_country).int().rename("zone"))                              # 0 = outside study area
peat_mask = lc.remap(PEAT_CLASS_CODES, [1] * len(PEAT_CLASS_CODES), 0).unmask(0).And(in_country)
study = zone.gt(0)

# tiles over the land-cover footprint ∩ country, keeping only those with study area
box = lc.geometry().bounds().intersection(gaul0.geometry().bounds(), 100).bounds()
ring = box.coordinates().getInfo()[0]
x0, x1 = math.floor(min(p[0] for p in ring)), math.ceil(max(p[0] for p in ring))
y0, y1 = math.floor(min(p[1] for p in ring)), math.ceil(max(p[1] for p in ring))
grid_tiles = [ee.Feature(ee.Geometry.Rectangle([x, y, x + TILE_DEG, y + TILE_DEG], None, False), {"tid": i})
              for i, (x, y) in enumerate((x, y) for x in np.arange(x0, x1, TILE_DEG) for y in np.arange(y0, y1, TILE_DEG))]
tiles_all = ee.FeatureCollection(grid_tiles)
counted = study.selfMask().reduceRegions(collection=tiles_all, reducer=ee.Reducer.count(), scale=500, tileScale=4)
tiles_fc = counted.filter(ee.Filter.gt("count", 0)).select(["tid"])
rects = [f["geometry"]["coordinates"][0] for f in tiles_fc.getInfo()["features"]]
xmin = min(p[0] for r in rects for p in r); xmax = max(p[0] for r in rects for p in r)
ymin = min(p[1] for r in rects for p in r); ymax = max(p[1] for r in rects for p in r)
EXTENT_BOX = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax], None, False)
STUDY_REGION = tiles_fc.geometry()

print(pd.DataFrame({"value": [", ".join(COUNTRY_LIST), len(grid_tiles), len(rects), f"{TILE_DEG}°",
                              f"{xmin:g}, {ymin:g}, {xmax:g}, {ymax:g}"]},
                   index=["country", "tiles over land cover", "tiles with study area (processed)", "tile size",
                          "extent (W, S, E, N)"]).to_string())

#### 5 · Disturbance layer: WUR driver alerts
The monthly driver maps are combined per pixel, keeping the most recent alert and its class. A pixel is a disturbance if its alert date falls in `START_YEAR`–`END_YEAR` and it lies in the study area.

In [ ]:
if BUILD_STACK:
    drv = dcol.map(lambda i: i.updateMask(i.select(DRIVER_BAND).gt(0))).qualityMosaic(DRIVER_DATE_BAND)
    drv_class, drv_date = drv.select(DRIVER_BAND).int(), drv.select(DRIVER_DATE_BAND).int()
    alert_year = drv_date.divide(1000).floor().add(2000).int()
    dist_mask = (drv_class.gt(0).And(alert_year.gte(START_YEAR)).And(alert_year.lte(END_YEAR))
                 .And(study)).selfMask().rename("dist")
    latest = drv_date.reduceRegion(ee.Reducer.max(), EXTENT_BOX, scale=500, maxPixels=1e10,
                                   bestEffort=True).getInfo().get(DRIVER_DATE_BAND)
else:   # latest alert stored in the shared stack
    latest = ee.Image(STACK_ASSET).select("t").reduceRegion(ee.Reducer.max(), EXTENT_BOX, scale=500, maxPixels=1e10,
                                                           bestEffort=True).getInfo().get("t")
LATEST_ALERT_DATE = yydoy_to_ts(latest).strftime("%Y-%m-%d") if latest else f"{END_YEAR}-12-31"

CLASS_CODES = sorted(DRIVER_CLASSES)
DRIVER_LABELS = ["unmapped code"] + [DRIVER_CLASSES[c] for c in CLASS_CODES]
ND = len(DRIVER_LABELS)
PARENT = {DRIVER_CLASSES[k]: DRIVER_CLASSES[v] for k, v in FIRE_SUBTYPES.items()}
if BUILD_STACK:
    drv_idx = drv_class.remap(CLASS_CODES, list(range(1, ND)), 0).rename("driver")

tbl = pd.DataFrame({"WUR code": CLASS_CODES, "label": [DRIVER_CLASSES[c] for c in CLASS_CODES],
                    "graph group": [PARENT.get(DRIVER_CLASSES[c], DRIVER_CLASSES[c]) for c in CLASS_CODES]})
def _swatch(row):
    col = DRIVER_GROUP_COLORS.get(row["graph group"])
    return [f"background-color: {col}" if c == "graph group" and col else "" for c in row.index]
print(f"Latest WUR alert in the study extent: {LATEST_ALERT_DATE}" + ("" if BUILD_STACK else " (from the shared stack)"))
try:
    display(tbl.set_index("WUR code").style.apply(_swatch, axis=1))
except Exception:
    display(tbl.set_index("WUR code"))

#### 6 · Distances to peat edge, roads and rivers
* **Peat edge:** precomputed `dist_to_edge_m` (distance inside the peat to its edge; 0 in the buffer).
* **Roads:** Kleinschroth et al. (2019). **Rivers:** HOTOSM waterway lines + water polygons. Both are painted, masked to the country, and turned into distance at 100 m.

In [ ]:
edge_km = ee.Image(PEAT_DIST_ASSET).select(PEAT_EDGE_BAND).divide(1000).unmask(0).rename("edge_km")

def painted(asset, width=None, flt=None):
    if asset_type(asset) != "TABLE":
        return None
    fc = ee.FeatureCollection(asset).filterBounds(EXTENT_BOX)
    if flt is not None:
        fc = fc.filter(flt)
    return ee.Image(0).byte().paint(fc, 1, width) if width else ee.Image(0).byte().paint(fc, 1)

road_src = painted(ROADS_ASSET, 1)
river_l = painted(RIVERS_LINES, 1, ee.Filter.inList(RIVER_TYPE_FIELD, RIVER_TYPES) if RIVER_TYPES else None)
water_p = painted(WATER_POLY)
river_src = None
for r in (river_l, water_p):
    if r is not None:
        river_src = r if river_src is None else river_src.Or(r)

FAR_KM = ACCESS_NEIGH_PX * ACCESS_SCALE / 1000
road_km = (distance_km(road_src.multiply(in_country), ACCESS_SCALE, ACCESS_NEIGH_PX) if road_src is not None
           else ee.Image(FAR_KM)).rename("road_km")
river_km = (distance_km(river_src.multiply(in_country), ACCESS_SCALE, ACCESS_NEIGH_PX) if river_src is not None
            else ee.Image(FAR_KM)).rename("river_km")

display(pd.DataFrame({"status": ["✓" if x is not None else "✗ missing → 'far' everywhere"
                                 for x in (road_src, river_l, water_p)]},
                     index=["roads", "river lines", "water polygons"]))

#### 6b · Protected areas, forest concessions, admin units
Each layer is painted with a polygon number (0 = none), masked to the country, so results can be reported per protected area, per concession and per admin unit. Polygons sharing a name are grouped.

In [ ]:
def id_layer(asset, preferred):
    """Paint polygons with an id (1..n, by distinct name); returns image, names, field."""
    if asset_type(asset) != "TABLE":
        return ee.Image(0).int(), [], None
    fc = ee.FeatureCollection(asset).filterBounds(EXTENT_BOX)
    f = name_field(fc, preferred)
    fc = fc.filter(ee.Filter.notNull([f]))
    names = [n for n in fc.aggregate_array(f).distinct().getInfo() if n not in (None, "")]
    if not names:
        return ee.Image(0).int(), [], f
    fc = fc.remap(names, list(range(1, len(names) + 1)), f)
    return ee.Image(0).int().paint(fc, f).multiply(in_country).int(), [str(n) for n in names], f

PA_IMG, PA_NAMES, PA_FIELD = id_layer(PA_ASSET, PA_NAME_FIELD)
CONC_IMG, CONC_NAMES, CONC_FIELD = id_layer(CONCESSIONS_ASSET, CONC_NAME_FIELD)
ADM_IMG, ADM_NAMES, ADM_FIELD = id_layer(ADMIN_ASSET, ADMIN_NAME_FIELD)
PA_IMG, CONC_IMG, ADM_IMG = PA_IMG.rename("pa"), CONC_IMG.rename("conc"), ADM_IMG.rename("adm")
display(pd.DataFrame({"name field": [PA_FIELD, CONC_FIELD, ADM_FIELD],
                      "polygons (distinct names) in extent": [len(PA_NAMES), len(CONC_NAMES), len(ADM_NAMES)]},
                     index=["protected areas", "forest concessions", "admin units"]).fillna("✗ not readable"))

#### 7 · Dry run on a small box (seconds)
Checks that everything evaluates before the exports.

In [ ]:
tb = ee.Geometry.Rectangle(TEST_BOX)
src_img = (ee.Image.pixelArea().divide(1e4).addBands(zone).addBands(drv_idx).updateMask(dist_mask) if BUILD_STACK else
           ee.Image.pixelArea().divide(1e4).addBands(ee.Image(STACK_ASSET).select("zone"))
           .addBands(ee.Image(STACK_ASSET).select("driver")).updateMask(ee.Image(STACK_ASSET).select("driver").mask()))
r = (src_img
     .reduceRegion(ee.Reducer.sum().group(1, "zone").group(2, "driver"), tb, scale=30, maxPixels=1e9).getInfo())
# bands: 0 area, 1 zone, 2 driver — outer group takes band 2 (driver), inner then groups the rest by band 1 (zone)
rows = [{"driver": DRIVER_LABELS[d["driver"]], "zone": ZONE_ORDER[z["zone"] - 1], "ha": z["sum"]}
        for d in r.get("groups", []) for z in d["groups"]]
if rows:
    display(pd.DataFrame(rows).pivot_table(index="driver", columns="zone", values="ha", aggfunc="sum").round(1))
else:
    print("No alerts in TEST_BOX — move it to a known hotspot")

#### 8 · Exports (two stages)
**8a · Stage 1 — stack (only with `OVERWRITE = True`, or if `STACK_ASSET` does not exist)** builds the shared 10 m stack of disturbed pixels: driver, alert date, class, protected area, concession, admin unit and the three distances (metres). This is the only heavy computation (≈1–2 h). Users skip it and read `STACK_ASSET`.
**8b · Stage 2 — outputs (everyone)** writes into `ASSET_ROOT`, re-exporting anything missing or older than the stack, and waits until they are ready:
* **stats**: disturbed area by year × month × driver × class × distance bins; by year × driver × class × protected area × concession × admin unit; and the study area of each combination (denominators);
* **events**: 8-connected disturbed pixels → one point per event with area, first date and modal driver, class, protected area, concession, admin unit and distance bins;
* **grid**: a table of ~1 km cells (`GRID_DEG`) with study area (ha) and disturbed area (ha) per year, computed as grouped sums per tile (no resampling, so it stays light).

##### 8a · Stage 1 — build the stack (only if `OVERWRITE = True` or it does not exist yet)

In [ ]:
ensure_folder(ASSET_ROOT)
STATS_ASSET = f"{ASSET_ROOT}/stats_{RUN_TAG}"
EVENTS_ASSET = f"{ASSET_ROOT}/events_{RUN_TAG}"
GRID_ASSET = f"{ASSET_ROOT}/grid1km_tbl_{RUN_TAG}"

if BUILD_STACK and STACK_EXISTS:                      # rebuild requested: needs write access to the stack
    try:
        ee.data.deleteAsset(STACK_ASSET)
        print(f"deleted the old stack {STACK_ASSET}")
    except Exception as e:
        print(f"⚠ cannot replace {STACK_ASSET} ({str(e)[:80]}) — keeping the existing stack")
        BUILD_STACK = False
if BUILD_STACK:
    stack_live = ee.Image.cat([
        drv_idx, drv_date.rename("t"), zone, PA_IMG, CONC_IMG, ADM_IMG,
        edge_km.multiply(1000).round().rename("edge_m"),
        road_km.multiply(1000).round().rename("road_m"),
        river_km.multiply(1000).round().rename("river_m")]).updateMask(dist_mask).toInt32()
    start_export("stack", lambda: ee.batch.Export.image.toAsset(
        image=stack_live, description=f"stack_{RUN_TAG}", assetId=STACK_ASSET, region=STUDY_REGION,
        scale=10, crs="EPSG:4326", maxPixels=1e13, pyramidingPolicy={".default": "sample"}),
        STACK_ASSET, overwrite=True)
else:
    t = asset_time(STACK_ASSET)
    print(f"Using shared stack {STACK_ASSET}" + (f" (updated {t:%Y-%m-%d %H:%M} UTC)" if t is not None else " — ⚠ not found"))

##### 8b · Stage 2 — stats, events and grid into your folder

In [ ]:
import time
def wait_for(assets, label):
    """Wait while the exports for `assets` are running (they keep running even if this kernel stops)."""
    t0 = time.time()
    while True:
        missing = [a for a in assets if asset_type(a) is None]
        if not missing:
            return True
        run = active_tasks()
        busy = [a for a in missing if a.split("/")[-1] in run]
        if not busy:
            print(f"\n⚠ {label}: {', '.join(a.split('/')[-1] for a in missing)} neither finished nor running — check the task list")
            return False
        if time.time() - t0 > WAIT_MAX_H * 3600:
            print(f"\n⚠ {label} still running after {WAIT_MAX_H} h — run this cell again later"); return False
        print(f"{pd.Timestamp.now():%H:%M}  waiting for {label}: {', '.join(a.split('/')[-1] for a in busy)} "
              f"— next check in {WAIT_POLL_MIN} min", end="\r")
        time.sleep(WAIT_POLL_MIN * 60)

if not wait_for([STACK_ASSET], "stack"):
    raise RuntimeError("The stack is not available — stage 2 cannot start")

NBE, NBA, NZ = len(EDGE_BINS_KM) + 1, len(ACCESS_BINS_KM) + 1, len(ZONE_CODES)
FIELDS = [("year", N_YEARS), ("month", 12), ("driver", ND), ("zone", NZ),
          ("edge_bin", NBE), ("road_bin", NBA), ("river_bin", NBA)]
NPA, NCO, NAD = len(PA_NAMES) + 1, len(CONC_NAMES) + 1, len(ADM_NAMES) + 1
PROT_FIELDS = [("year", N_YEARS), ("driver", ND), ("zone", NZ), ("pa", NPA), ("conc", NCO), ("adm", NAD)]
ZP_FIELDS = [("zone", NZ), ("pa", NPA), ("conc", NCO), ("adm", NAD)]
for nm, f in [("main", FIELDS), ("protection/admin", PROT_FIELDS)]:
    cap = int(np.prod([s for _, s in f]))
    assert cap < 2**31, f"{nm} code space {cap:,} too large — too many polygons; group them by a coarser name field"

if True:
    st = ee.Image(STACK_ASSET)
    t = st.select("t"); yr = t.divide(1000).floor().add(2000).int()
    bands = {"year": yr.subtract(START_YEAR), "month": doy_to_month(yr, t.mod(1000).int()).subtract(1),
             "driver": st.select("driver"), "zone": st.select("zone").subtract(1),
             "edge_bin": bin_index(st.select("edge_m").divide(1000), EDGE_BINS_KM),
             "road_bin": bin_index(st.select("road_m").divide(1000), ACCESS_BINS_KM),
             "river_bin": bin_index(st.select("river_m").divide(1000), ACCESS_BINS_KM),
             "pa": st.select("pa"), "conc": st.select("conc"), "adm": st.select("adm")}
    dmask = st.select("driver").mask()
    area = ee.Image.pixelArea().divide(1e4).rename("area")
    dist_stack = area.addBands(encode(bands, FIELDS).rename("code")).updateMask(dmask)
    prot_stack = area.addBands(encode(bands, PROT_FIELDS).rename("code")).updateMask(dmask)
    zone_stack = area.addBands(encode({"zone": zone.subtract(1), "pa": PA_IMG, "conc": CONC_IMG, "adm": ADM_IMG},
                                      ZP_FIELDS).rename("code")).updateMask(study)

    # stats: disturbed area by code (10 m) + zone areas (land-cover resolution)
    lc_scale = LC_PROJ.nominalScale().getInfo()
    def reduce_tile(tile):
        g, tid = tile.geometry(), tile.get("tid")
        grp = ee.Reducer.sum().group(1, "code")
        to_fc = lambda groups, kind: ee.FeatureCollection(ee.List(groups).map(
            lambda d: ee.Feature(g.centroid(100), ee.Dictionary(d)).set({"kind": kind, "tid": tid})))
        rd = dist_stack.reduceRegion(grp, g, scale=10, maxPixels=1e13, tileScale=TILE_SCALE)
        rp = prot_stack.reduceRegion(grp, g, scale=10, maxPixels=1e13, tileScale=TILE_SCALE)
        rz = zone_stack.reduceRegion(grp, g, scale=lc_scale, maxPixels=1e13, tileScale=TILE_SCALE)
        return (to_fc(rd.get("groups"), "dist").merge(to_fc(rp.get("groups"), "prot"))
                .merge(to_fc(rz.get("groups"), "zone")))
    start_export("stats", lambda: ee.batch.Export.table.toAsset(
        collection=tiles_fc.map(reduce_tile).flatten(), description=f"stats_{RUN_TAG}", assetId=STATS_ASSET), STATS_ASSET, newer_than=STACK_ASSET)

    # events
    EV_CATS = ["driver", "zone", "edge_bin", "road_bin", "river_bin", "pa", "conc", "adm"]
    ev_img = (ee.Image(1).int().rename("seg").addBands(t.rename("t_first"))
              .addBands([ee.Image(bands[k]).int().rename(k) for k in EV_CATS]).updateMask(dmask))
    ev_red = ee.Reducer.min().unweighted().setOutputs(["t_first"])
    for k in EV_CATS:
        ev_red = ev_red.combine(ee.Reducer.mode().unweighted().setOutputs([k]), "", False)
    def tile_events(tile):
        v = ev_img.reduceToVectors(reducer=ev_red, geometry=tile.geometry(), scale=10, geometryType="polygon",
                                   eightConnected=True, labelProperty="seg", maxPixels=1e11, tileScale=TILE_SCALE)
        return v.map(lambda f: ee.Feature(f.geometry().centroid(1), f.toDictionary(["t_first"] + EV_CATS))
                     .set({"area_ha": f.geometry().area(1).divide(1e4)}))
    start_export("events", lambda: ee.batch.Export.table.toAsset(
        collection=tiles_fc.map(tile_events).flatten(), description=f"events_{RUN_TAG}", assetId=EVENTS_ASSET), EVENTS_ASSET, newer_than=STACK_ASSET)

    # 1 km grid as a table: grouped sums per (cell, year) in each tile — no resampling, low memory
    NCOLS, NROWS = int(math.ceil((xmax - xmin) / GRID_DEG)), int(math.ceil((ymax - ymin) / GRID_DEG))
    assert NCOLS * NROWS * N_YEARS < 2**31, "grid too large — increase GRID_DEG"
    ll = ee.Image.pixelLonLat()
    cell = (ll.select("latitude").multiply(-1).add(ymax).divide(GRID_DEG).floor().multiply(NCOLS)
            .add(ll.select("longitude").subtract(xmin).divide(GRID_DEG).floor())).int()
    g_dist = area.addBands(cell.multiply(N_YEARS).add(yr.subtract(START_YEAR)).rename("code")).updateMask(dmask)
    g_study = area.addBands(cell.rename("code")).updateMask(study)
    def grid_tile(tile):
        g, tid = tile.geometry(), tile.get("tid")
        grp = ee.Reducer.sum().group(1, "code")
        to_fc = lambda groups, kind: ee.FeatureCollection(ee.List(groups).map(
            lambda d: ee.Feature(g.centroid(100), ee.Dictionary(d)).set({"kind": kind})))
        rd = g_dist.reduceRegion(grp, g, scale=10, maxPixels=1e13, tileScale=TILE_SCALE)
        rs = g_study.reduceRegion(grp, g, scale=lc_scale, maxPixels=1e13, tileScale=TILE_SCALE)
        return to_fc(rd.get("groups"), "dist").merge(to_fc(rs.get("groups"), "study"))
    start_export("grid", lambda: ee.batch.Export.table.toAsset(
        collection=tiles_fc.map(grid_tile).flatten(), description=f"grid1km_tbl_{RUN_TAG}", assetId=GRID_ASSET), GRID_ASSET, newer_than=STACK_ASSET)

    if wait_for([STATS_ASSET, EVENTS_ASSET, GRID_ASSET], "outputs"):
        print("\n✓ stats, events and grid are ready — continue with Part B")

#### 9 · Status
The tasks of this run that are still active (also those started before a kernel restart), and the state of the stack and your outputs.

In [ ]:
run = active_tasks()   # includes tasks started before a kernel restart
mine = {d: v for d, v in run.items() if d and RUN_TAG in d}
if mine:
    display(pd.DataFrame([{"task": d, "state": v[0], "id": v[1]} for d, v in mine.items()]).set_index("task"))
if TASKS:
    display(pd.DataFrame([{"export": n, "state": t.status().get("state"), "error": t.status().get("error_message", "")}
                          for n, t in TASKS.items()]).set_index("export"))
display(pd.DataFrame({"asset": [STACK_ASSET, STATS_ASSET, EVENTS_ASSET, GRID_ASSET],
                      "ready": ["✓" if asset_type(a) else "…" for a in (STACK_ASSET, STATS_ASSET, EVENTS_ASSET, GRID_ASSET)],
                      "updated (UTC)": [f"{t:%Y-%m-%d %H:%M}" if t is not None else "–"
                                        for t in map(asset_time, (STACK_ASSET, STATS_ASSET, EVENTS_ASSET, GRID_ASSET))]},
                     index=["shared stack", "stats", "events", "grid"]))

### Part B · Analysis
#### 10 · Load results
Each table is downloaded once as CSV (only the needed columns) and cached in `outputs_peat_disturbance/ee_tables/`; later runs read the local copy. Set `REFRESH_TABLES = True` in §1 to download them again after re-exporting.

In [ ]:
raw = table_df(STATS_ASSET, ["kind", "code", "sum"])
d = raw[raw.kind == "dist"]
D = decode(d["code"], FIELDS); D["area_ha"] = d["sum"].to_numpy()
D = D.groupby([n for n, _ in FIELDS], as_index=False).area_ha.sum()
D["year"] = D.year + START_YEAR; D["month"] = D.month + 1
D["driver"] = D.driver.map(dict(enumerate(DRIVER_LABELS)))
D["zone"] = D.zone.map(dict(enumerate(ZONE_ORDER)))
D["access_bin"] = D[["road_bin", "river_bin"]].min(axis=1)       # nearest of road or river
NAMES = {"pa": ["— outside PA"] + PA_NAMES, "conc": ["— outside concession"] + CONC_NAMES,
         "adm": ["— no admin unit"] + ADM_NAMES}
def unpack(kind, fields):
    x = raw[raw.kind == kind]
    df = decode(x["code"], fields); df["area_ha"] = x["sum"].to_numpy()
    df = df.groupby([n for n, _ in fields], as_index=False).area_ha.sum()
    if "year" in df: df["year"] = df.year + START_YEAR
    if "driver" in df: df["driver"] = df.driver.map(dict(enumerate(DRIVER_LABELS)))
    df["zone"] = df.zone.map(dict(enumerate(ZONE_ORDER)))
    for k in ("pa", "conc", "adm"):
        if k in df: df[k] = df[k].map(dict(enumerate(NAMES[k])))
    return df
P = unpack("prot", PROT_FIELDS)          # disturbed area by protection / admin unit
ZP = unpack("zone", ZP_FIELDS)           # study area by protection / admin unit
Z = ZP.groupby("zone").area_ha.sum().reindex(ZONE_ORDER)

EV = table_df(EVENTS_ASSET, ["t_first", "driver", "zone", "edge_bin", "road_bin", "river_bin",
                              "pa", "conc", "adm", "area_ha", ".geo"])
EV_COLS = ["driver", "zone", "edge_bin", "road_bin", "river_bin", "pa", "conc", "adm"]
miss = EV[EV_COLS].isna().sum()
if miss.any():
    print("Events with a missing category (kept, shown as missing in the tables):"); display(miss[miss > 0].to_frame("events"))
for c in EV_COLS:
    EV[c] = EV[c].round().astype("Int64")
for k in ("pa", "conc", "adm"):
    EV[k] = EV[k].map(dict(enumerate(NAMES[k])))
EV["driver"] = EV.driver.map(dict(enumerate(DRIVER_LABELS)))
EV["zone"] = (EV.zone).map(dict(enumerate(ZONE_ORDER)))
EV["access_bin"] = EV[["road_bin", "river_bin"]].min(axis=1)
EV = EV[EV.t_first.notna()].copy()
EV["date"] = EV.t_first.apply(yydoy_to_ts)
EV["year"], EV["month"] = EV.date.dt.year, EV.date.dt.month
SIZE_LABELS = [f"small (<{SIZE_CLASSES_HA[0]:g} ha)", f"medium ({SIZE_CLASSES_HA[0]:g}–{SIZE_CLASSES_HA[1]:g} ha)",
               f"large (≥{SIZE_CLASSES_HA[1]:g} ha)"]
EV["size_class"] = pd.cut(EV.area_ha, [0] + SIZE_CLASSES_HA + [np.inf], right=False, labels=SIZE_LABELS)
EV["below_mmu"] = EV.area_ha < MMU_HA

for df in (D, EV, P):
    df["with_fire"] = df.driver.isin(PARENT)
    df["driver_group"] = df.driver.map(lambda d: PARENT.get(d, d))
GROUPS = [g for g in DRIVER_GROUP_COLORS if g in set(D.driver_group)]
GCOL = [DRIVER_GROUP_COLORS[g] for g in GROUPS]
ZCOL = [ZONE_COLORS[z] for z in ZONE_ORDER]
PROVISIONAL_FROM = (pd.Timestamp(LATEST_ALERT_DATE) - pd.DateOffset(months=3)).to_period("M")
FULL_YEARS = [y for y in YEARS if pd.Period(f"{y}-12", "M") < PROVISIONAL_FROM]

D.to_csv(LOCAL_OUT / f"disturbance_stats_{RUN_TAG}.csv", index=False)
P.to_csv(LOCAL_OUT / f"disturbance_by_pa_concession_admin_{RUN_TAG}.csv", index=False)
EV.to_csv(LOCAL_OUT / f"events_{RUN_TAG}.csv", index=False)
print(f"{D.area_ha.sum():,.0f} ha disturbed in {Z.sum():,.0f} ha of study area · {len(EV):,} events · "
      f"latest alert {LATEST_ALERT_DATE} · complete years {FULL_YEARS}")

#### 11 · Disturbance and drivers by land-cover class

In [ ]:
area_tbl = pd.DataFrame({"class area (ha)": Z, "disturbed (ha)": D.groupby("zone").area_ha.sum().reindex(ZONE_ORDER)})
area_tbl["% of class disturbed"] = area_tbl["disturbed (ha)"] / area_tbl["class area (ha)"] * 100
area_tbl["share of disturbance (%)"] = pct(area_tbl["disturbed (ha)"])
area_tbl.loc["total"] = [area_tbl.iloc[:, 0].sum(), area_tbl.iloc[:, 1].sum(),
                         area_tbl.iloc[:, 1].sum() / area_tbl.iloc[:, 0].sum() * 100, 100]
print("Study area and disturbance by class"); display(area_tbl.round(2))

dz = D.pivot_table(index="driver", columns="zone", values="area_ha", aggfunc="sum", fill_value=0).reindex(columns=ZONE_ORDER, fill_value=0)
dz["total"] = dz.sum(axis=1)
dz = dz.sort_values("total", ascending=False)
print("Disturbed area (ha) by driver and class"); display(dz.round(1))
print("Share of each class's disturbance (%)"); display((dz[ZONE_ORDER] / dz[ZONE_ORDER].sum() * 100).round(1))

dg = D.pivot_table(index="zone", columns="driver_group", values="area_ha", aggfunc="sum", fill_value=0).reindex(index=ZONE_ORDER, columns=GROUPS, fill_value=0)
dy = D.pivot_table(index=["zone", "year"], columns="driver_group", values="area_ha", aggfunc="sum", fill_value=0).reindex(columns=GROUPS, fill_value=0)
n_pan = 2 + len(ZONE_ORDER)
fig, axes = plt.subplots(math.ceil(n_pan / 2), 2, figsize=(16, 5.5 * math.ceil(n_pan / 2)), squeeze=False)
axes = axes.flatten()
dg.plot.bar(stacked=True, ax=axes[0], color=GCOL, legend=False, rot=15)
axes[0].set_ylabel("Disturbed area (ha)"); axes[0].set_title("Area by class and driver")
dg.div(dg.sum(axis=1), axis=0).mul(100).plot.bar(stacked=True, ax=axes[1], color=GCOL, legend=False, rot=15)
axes[1].set_ylabel("% of class disturbance"); axes[1].set_title("Driver shares")
for ax, z in zip(axes[2:], ZONE_ORDER):
    if z in dy.index.get_level_values(0):
        dy.loc[z].plot.bar(stacked=True, ax=ax, color=GCOL, legend=False, rot=0)
    ax.set_title(f"{z} — by year"); ax.set_ylabel("Disturbed area (ha)"); ax.set_xlabel("")
for ax in axes[:2]:
    ax.set_xlabel("")
handles = [Patch(color=c, label=g) for g, c in zip(GROUPS, GCOL)]
if n_pan % 2:                                   # spare panel → legend
    axes[-1].axis("off"); axes[-1].legend(handles=handles, loc="center", fontsize=11, title="Driver", frameon=False)
else:
    fig.legend(handles=handles, loc="lower center", ncol=min(len(GROUPS), 5), bbox_to_anchor=(0.5, -0.04), fontsize=10)
plt.tight_layout(); save(fig, "drivers_by_class"); plt.show()

fire = D[D.driver_group.isin(PARENT.values())]
if len(fire):
    print("Agricultural clearing flagged 'with fire' (% of that group's area, by class)")
    display((fire[fire.with_fire].groupby(["driver_group", "zone"]).area_ha.sum()
             / fire.groupby(["driver_group", "zone"]).area_ha.sum() * 100).unstack().reindex(columns=ZONE_ORDER).round(1))

#### 12 · Protected areas, forest concessions and admin units
Area basis. 'Study area' is the area of the three land-cover classes inside each polygon.

In [ ]:
def status(df):
    return np.select([(df.pa != NAMES["pa"][0]) & (df.conc != NAMES["conc"][0]), df.pa != NAMES["pa"][0],
                      df.conc != NAMES["conc"][0]], ["PA + concession", "protected area", "concession"], "neither")
P["status"], ZP["status"], EV["status"] = status(P), status(ZP), status(EV)
STATUS_COLORS = {"neither": "#d9d9d9", "protected area": "#6a51a3", "concession": "#bf812d", "PA + concession": "#e7298a"}
SORDER = [s for s in STATUS_COLORS if s in set(ZP.status)]

st_tbl = pd.DataFrame({"study area (ha)": ZP.groupby("status").area_ha.sum(),
                       "disturbed (ha)": P.groupby("status").area_ha.sum()}).reindex(SORDER).fillna(0)
st_tbl["% of study area disturbed"] = st_tbl["disturbed (ha)"] / st_tbl["study area (ha)"] * 100
st_tbl["share of disturbance (%)"] = pct(st_tbl["disturbed (ha)"])
st_tbl["share of events (%)"] = pct(EV.status.value_counts()).reindex(SORDER)
print("Disturbance by protection status"); display(st_tbl.round(2))
print("% of each class disturbed, by protection status")
display((P.pivot_table(index="status", columns="zone", values="area_ha", aggfunc="sum")
         / ZP.pivot_table(index="status", columns="zone", values="area_ha", aggfunc="sum") * 100)
        .reindex(index=SORDER, columns=ZONE_ORDER).round(2))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sz = P.pivot_table(index="zone", columns="status", values="area_ha", aggfunc="sum", fill_value=0).reindex(index=ZONE_ORDER, columns=SORDER, fill_value=0)
sz.div(sz.sum(axis=1), axis=0).mul(100).plot.barh(stacked=True, ax=axes[0], color=[STATUS_COLORS[c] for c in SORDER])
axes[0].set_xlabel("% of disturbed area"); axes[0].set_title("Protection status of disturbance, by class")
sd_ = P.pivot_table(index="status", columns="driver_group", values="area_ha", aggfunc="sum", fill_value=0).reindex(index=SORDER, columns=GROUPS, fill_value=0)
sd_.div(sd_.sum(axis=1), axis=0).mul(100).plot.barh(stacked=True, ax=axes[1], color=GCOL)
axes[1].set_xlabel("% of disturbed area"); axes[1].set_title("Drivers inside / outside PAs and concessions")
axes[1].legend(bbox_to_anchor=(1, 1), fontsize=7)
plt.tight_layout(); save(fig, "protection_status"); plt.show()

def per_polygon(key, what, fname, n=15):
    dist = P[P[key] != NAMES[key][0]]
    if dist.empty and ZP[ZP[key] != NAMES[key][0]].empty:
        print(f"No {what} in the study area"); return None
    t = pd.DataFrame({"study area (ha)": ZP[ZP[key] != NAMES[key][0]].groupby(key).area_ha.sum(),
                      "disturbed (ha)": dist.groupby(key).area_ha.sum()}).fillna(0)
    t["% disturbed"] = t["disturbed (ha)"] / t["study area (ha)"].replace(0, np.nan) * 100
    top = dist.groupby([key, "driver_group"]).area_ha.sum().reset_index().sort_values("area_ha").drop_duplicates(key, keep="last")
    t["main driver"] = top.set_index(key).driver_group.reindex(t.index).fillna("–")
    t = t.sort_values("disturbed (ha)", ascending=False)
    t.to_csv(LOCAL_OUT / f"{fname}_{RUN_TAG}.csv")
    print(f"Disturbance by {what}"); display(t.round(2))
    sub = t.head(n).index
    bars = (dist[dist[key].isin(sub)].pivot_table(index=key, columns="driver_group", values="area_ha", aggfunc="sum",
            fill_value=0).reindex(index=sub[::-1], columns=GROUPS, fill_value=0))
    fig, axes = plt.subplots(1, 2, figsize=(15, max(3, 0.35 * len(sub) + 1)), sharey=True)
    bars.plot.barh(stacked=True, ax=axes[0], color=GCOL, legend=False)
    axes[0].set_xlabel("Disturbed area (ha)"); axes[0].set_title(f"Top {len(sub)} {what} by disturbed area"); axes[0].set_ylabel("")
    t.loc[sub[::-1], "% disturbed"].plot.barh(ax=axes[1], color="#d7191c")
    axes[1].set_xlabel("% of study area disturbed"); axes[1].set_title("Relative to study area")
    fig.legend(handles=[Patch(color=c, label=g) for g, c in zip(GROUPS, GCOL)], loc="lower center",
               ncol=min(len(GROUPS), 5), bbox_to_anchor=(0.5, -0.12), fontsize=7)
    plt.tight_layout(); save(fig, fname); plt.show()
    return t

PA_T = per_polygon("pa", "protected areas", "by_protected_area")
CONC_T = per_polygon("conc", "forest concessions", "by_concession")
ADM_T = per_polygon("adm", "admin units", "by_admin_unit", n=20)

#### 13 · Event sizes
Events are 8-connected disturbed pixels; size classes as in Nesha et al. (small < 0.5 ha, medium 0.5–2 ha, large ≥ 2 ha).

In [ ]:
summ = EV.groupby("zone").area_ha.describe(percentiles=[.5, .9]).reindex(ZONE_ORDER)
summ.loc["all"] = EV.area_ha.describe(percentiles=[.5, .9])
print(f"{len(EV):,} events · {EV.below_mmu.mean() * 100:.1f} % below {MMU_HA} ha (fragments)")
display(summ.rename(columns={"count": "events", "mean": "mean ha", "50%": "median ha", "90%": "p90 ha", "max": "max ha"})
        [["events", "mean ha", "median ha", "p90 ha", "max ha"]].round(2))

sc = EV.groupby("size_class", observed=False).agg(events=("area_ha", "size"), area_ha=("area_ha", "sum"))
sc["% of events"] = pct(sc.events); sc["% of area"] = pct(sc.area_ha)
display(sc.round(1))
print("Event-size classes by driver (% of each driver's area)")
display(pd.crosstab(EV.driver_group, EV.size_class, values=EV.area_ha, aggfunc="sum", normalize="index")
        .mul(100).reindex(GROUPS).round(1))

fig, axes = plt.subplots(3, 1, figsize=(12, 16))
bins = np.logspace(np.log10(max(EV.area_ha.min(), 0.01)), np.log10(EV.area_ha.max() + 0.01), 40)
for z, c in zip(ZONE_ORDER, ZCOL):
    axes[0].hist(EV.loc[EV.zone == z, "area_ha"], bins=bins, histtype="step", lw=1.5, color=c, label=z)
axes[0].set_xscale("log"); plain_log(axes[0].xaxis); axes[0].set_xlabel("Event area (ha, log scale)"); axes[0].set_ylabel("Events"); axes[0].legend(fontsize=8)
axes[0].set_title("Event size distribution")
for t in SIZE_CLASSES_HA: axes[0].axvline(t, color="grey", ls=":")
sd = EV.pivot_table(index="size_class", columns="driver_group", values="area_ha", aggfunc="sum", observed=False,
                    fill_value=0).reindex(columns=GROUPS, fill_value=0)
sd.div(sd.sum(axis=1), axis=0).mul(100).plot.bar(stacked=True, ax=axes[1], color=GCOL, rot=0)
axes[1].set_ylabel("% of area"); axes[1].set_xlabel(""); axes[1].set_title("Driver shares by event size"); axes[1].legend(fontsize=7)
EV.boxplot(column="area_ha", by="driver_group", ax=axes[2], rot=30, showfliers=False)
axes[2].set_yscale("log"); plain_log(axes[2].yaxis); axes[2].set_ylabel("Event area (ha, log scale)"); axes[2].set_title("Event size by driver"); axes[2].set_xlabel("")
plt.suptitle(""); plt.tight_layout(); save(fig, "event_sizes"); plt.show()

#### 14 · Temporal patterns by month
Share of each year's disturbance per month; months above `TEMPORAL_HOTSPOT_PCT` are temporal hotspots. Dashed lines are partial or provisional years.

In [ ]:
mt = D.pivot_table(index="year", columns="month", values="area_ha", aggfunc="sum", fill_value=0).reindex(columns=range(1, 13), fill_value=0)
mp = mt.div(mt.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for y in mt.index:
    ls = "-" if y in FULL_YEARS else "--"
    axes[0].plot(range(1, 13), mt.loc[y], marker="o", ls=ls, label=str(y))
    axes[1].plot(range(1, 13), mp.loc[y], marker="o", ls=ls, label=str(y))
axes[1].axhline(TEMPORAL_HOTSPOT_PCT, color="grey", ls=":")
axes[0].set_ylabel("Disturbed area (ha)"); axes[1].set_ylabel("% of the year's disturbance")
for a in axes: a.set_xticks(range(1, 13)); a.set_xlabel("Month")
axes[0].legend(ncol=3, fontsize=8); plt.tight_layout(); save(fig, "monthly"); plt.show()

fig, ax = plt.subplots(figsize=(10, 0.6 * len(mp) + 1))
im = ax.imshow(mp.to_numpy(), aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(12)); ax.set_xticklabels(range(1, 13)); ax.set_yticks(range(len(mp))); ax.set_yticklabels(mp.index)
for (i, j), v in np.ndenumerate(mp.to_numpy()):
    if v > TEMPORAL_HOTSPOT_PCT: ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="% of year"); ax.set_title(f"Temporal hotspots (months > {TEMPORAL_HOTSPOT_PCT} % labelled)")
plt.tight_layout(); save(fig, "monthly_heatmap"); plt.show()

display(pd.DataFrame({"Jan–Jun share (%)": mp[list(range(1, 7))].sum(axis=1).round(1),
                      "hotspot months": [[m for m in range(1, 13) if mp.loc[y, m] > TEMPORAL_HOTSPOT_PCT] for y in mp.index]},
                     index=mp.index))

# monthly profile per driver (all years pooled) and per class
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
pg = D[D.year.isin(FULL_YEARS)].pivot_table(index="month", columns="driver_group", values="area_ha", aggfunc="sum",
                                             fill_value=0).reindex(index=range(1, 13), columns=GROUPS, fill_value=0)
(pg / pg.sum() * 100).plot(ax=axes[0], color=GCOL, marker=".")
axes[0].set_ylabel("% of driver's annual area"); axes[0].set_title("Monthly profile by driver (complete years)")
pz = D[D.year.isin(FULL_YEARS)].pivot_table(index="month", columns="zone", values="area_ha", aggfunc="sum",
                                             fill_value=0).reindex(index=range(1, 13), columns=ZONE_ORDER, fill_value=0)
(pz / pz.sum() * 100).plot(ax=axes[1], color=ZCOL, marker=".")
axes[1].set_ylabel("% of class's annual area"); axes[1].set_title("Monthly profile by class (complete years)")
for a in axes: a.set_xticks(range(1, 13)); a.legend(fontsize=7)
plt.tight_layout(); save(fig, "monthly_by_driver_class"); plt.show()

#### 15 · Spatial hotspots on the 1 km grid
Intensity = disturbed area / study area in each ~1 km cell (`GRID_DEG`, whole period). Low < 2 %, medium 2–20 %, high > 20 % (spatial hotspot), as in Nesha et al. Followed by maps of the number of events and the median event size per cell, and the dominant driver.

In [ ]:
lo, hi = INTENSITY_THRESHOLDS_PCT
NCOLS, NROWS = int(math.ceil((xmax - xmin) / GRID_DEG)), int(math.ceil((ymax - ymin) / GRID_DEG))
GEXT = (xmin, xmin + NCOLS * GRID_DEG, ymax - NROWS * GRID_DEG, ymax)
GT = table_df(GRID_ASSET, ["kind", "code", "sum"])
gd = GT[GT.kind == "dist"]
dist_cy = (pd.DataFrame({"cell": (gd.code.round().astype("int64") // N_YEARS),
                         "year": (gd.code.round().astype("int64") % N_YEARS) + START_YEAR, "ha": gd["sum"]})
           .groupby(["cell", "year"]).ha.sum().unstack(fill_value=0).reindex(columns=YEARS, fill_value=0))
study_c = GT[GT.kind == "study"].assign(cell=lambda x: x.code.round().astype("int64")).groupby("cell")["sum"].sum()
CELLS = pd.DataFrame({"study_ha": study_c}).join(dist_cy, how="outer").fillna(0)
CELLS["dist_ha_all"] = CELLS[YEARS].sum(axis=1)
CELLS["intensity_pct"] = (CELLS.dist_ha_all / CELLS.study_ha.where(CELLS.study_ha > 0) * 100).clip(upper=100)
CELLS["class"] = pd.cut(CELLS.intensity_pct, [0, lo, hi, np.inf], right=False,
                        labels=[f"low (<{lo} %)", f"medium ({lo}–{hi} %)", f"high / hotspot (>{hi} %)"])
CELLS.loc[CELLS.dist_ha_all <= 0, "class"] = np.nan
CELLS["lon"] = xmin + (CELLS.index % NCOLS + 0.5) * GRID_DEG
CELLS["lat"] = ymax - (CELLS.index // NCOLS + 0.5) * GRID_DEG
CELLS.to_csv(LOCAL_OUT / f"grid1km_{RUN_TAG}.csv")
CLS_NAMES = list(CELLS["class"].cat.categories); CLS_COLORS = ["#fdae61", "#d7191c", "#2c007a"]

INT = CELLS.dropna(subset=["class"]).groupby("class", observed=False).agg(**{"1 km cells": ("dist_ha_all", "size"),
                                                                             "disturbed ha": ("dist_ha_all", "sum")})
INT["% of cells"] = pct(INT["1 km cells"]); INT["% of disturbed area"] = pct(INT["disturbed ha"])
print(f"{(CELLS.dist_ha_all > 0).sum():,} disturbed cells of {(CELLS.study_ha > 0).sum():,} cells with study area")
display(INT.round(1))
print("Top 15 hotspot cells")
display(CELLS.sort_values("intensity_pct", ascending=False).head(15)[["lon", "lat", "study_ha", "dist_ha_all", "intensity_pct"] + YEARS].round(2))

def to_arr(series):
    a = np.full(NROWS * NCOLS, np.nan)
    v = series.dropna(); v = v[(v.index >= 0) & (v.index < NROWS * NCOLS)]
    a[v.index.to_numpy()] = v.to_numpy()
    return a.reshape(NROWS, NCOLS)
study_arr = to_arr(CELLS.study_ha.where(CELLS.study_ha > 0))
cls_arr = to_arr(CELLS["class"].cat.codes.where(CELLS["class"].notna()).astype(float) + 1)
EXTENT = GEXT

borders = [r for f in gaul0.map(lambda f: f.simplify(2000)).getInfo()["features"] for r in geom_rings(f["geometry"])]
def outlines(asset):
    if asset_type(asset) != "TABLE":
        return []
    gj = ee.FeatureCollection(asset).filterBounds(EXTENT_BOX).map(lambda f: ee.Feature(f.geometry().simplify(500))).getInfo()
    return [r for f in gj["features"] for r in geom_rings(f["geometry"])]
PA_LINES, CONC_LINES = outlines(PA_ASSET), outlines(CONCESSIONS_ASSET)
def basemap(ax, title, context=True):
    for b in borders: ax.plot(b[:, 0], b[:, 1], color="black", lw=0.6)
    if context:
        for l in PA_LINES: ax.plot(l[:, 0], l[:, 1], color="#6a51a3", lw=0.7)
        for l in CONC_LINES: ax.plot(l[:, 0], l[:, 1], color="#8c510a", lw=0.4, ls="--")
    ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax); ax.set_aspect("equal"); ax.set_title(title, fontsize=10)
    ax.tick_params(labelsize=7); ax.set_facecolor("#f7f7f7")

fig, ax = plt.subplots(figsize=(11, 8))
ax.imshow(np.where(study_arr > 0, 1, np.nan), extent=EXTENT, cmap=ListedColormap(["#d9f0d3"]), interpolation="nearest")
ax.imshow(cls_arr, extent=EXTENT, cmap=ListedColormap(CLS_COLORS), vmin=0.5, vmax=3.5, interpolation="nearest")
basemap(ax, f"Disturbance intensity per ~1 km cell, {START_YEAR}–{END_YEAR}")
ax.legend(handles=[Patch(color="#d9f0d3", label="study area, no disturbance")] +
          [Patch(color=c, label=n) for n, c in zip(CLS_NAMES, CLS_COLORS)] +
          [Line2D([], [], color="#6a51a3", lw=0.9, label="protected area"),
           Line2D([], [], color="#8c510a", lw=0.7, ls="--", label="forest concession")], loc="lower left", fontsize=8)
plt.tight_layout(); save(fig, "map_hotspots"); plt.show()

ncol = min(3, N_YEARS); nrow = math.ceil(N_YEARS / ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(5.5 * ncol, 4.5 * nrow), squeeze=False)
for ax, y in zip(axes.flat, YEARS):
    sh = to_arr((CELLS[y] / CELLS.study_ha.where(CELLS.study_ha > 0) * 100).where(CELLS[y] > 0))
    ax.imshow(np.where(study_arr > 0, 1, np.nan), extent=EXTENT, cmap=ListedColormap(["#e5f5e0"]))
    im = ax.imshow(sh, extent=EXTENT, cmap="inferno_r", vmin=0, vmax=hi / 2, interpolation="nearest")
    basemap(ax, f"{y}" + ("" if y in FULL_YEARS else " (partial)"), context=False)
for ax in axes.flat[N_YEARS:]: ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.5, label="% of study area disturbed in the year")
save(fig, "map_hotspots_by_year"); plt.show()

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(CELLS.intensity_pct[CELLS.dist_ha_all > 0], bins=np.arange(0, 101, 1), color="teal")
for t in INTENSITY_THRESHOLDS_PCT: ax.axvline(t, color="red", ls="--", lw=0.8)
ax.set_yscale("log"); plain_log(ax.yaxis); ax.set_xlabel("Intensity (% of study area disturbed per ~1 km cell)"); ax.set_ylabel("Cells (log scale)")
plt.tight_layout(); save(fig, "hist_intensity"); plt.show()

# events on the ~1 km grid: number of events and median event size per cell
from matplotlib.colors import LogNorm
col_i = np.floor((EV.lon - xmin) / GRID_DEG); row_i = np.floor((ymax - EV.lat) / GRID_DEG)
inside = col_i.between(0, NCOLS - 1) & row_i.between(0, NROWS - 1)          # events inside the grid extent
evg = EV[inside].assign(cell=(row_i[inside] * NCOLS + col_i[inside]).astype("int64")).groupby("cell").agg(events=("area_ha", "size"), median_ha=("area_ha", "median"),
                                                 total_ha=("area_ha", "sum"))
print(f"Events per ~1 km cell: median {evg.events.median():.0f}, max {evg.events.max():,} · "
      f"{(evg.events >= 10).sum():,} cells with ≥ 10 events")
for col, title, cmap, norm, lab in [
        ("events", "Number of disturbance events per ~1 km cell", "magma_r", LogNorm(1, max(evg.events.max(), 2)), "events (log scale)"),
        ("median_ha", "Median event size per ~1 km cell", "viridis_r", LogNorm(max(evg.median_ha.min(), 0.01), max(evg.median_ha.max(), 0.1)), "median event size (ha, log scale)")]:
    fig, ax = plt.subplots(figsize=(11, 8))
    ax.imshow(np.where(study_arr > 0, 1, np.nan), extent=EXTENT, cmap=ListedColormap(["#e5f5e0"]), interpolation="nearest")
    im = ax.imshow(to_arr(evg[col]), extent=EXTENT, cmap=cmap, norm=norm, interpolation="nearest")
    basemap(ax, f"{title}, {START_YEAR}–{END_YEAR}")
    cb = fig.colorbar(im, ax=ax, shrink=0.6, label=lab); plain_log(cb.ax.yaxis)
    plt.tight_layout(); save(fig, f"map_{col}_per_cell"); plt.show()

# dominant driver per DOM_CELL_DEG cell (from events)
nx, ny = int(round((xmax - xmin) / DOM_CELL_DEG)), int(round((ymax - ymin) / DOM_CELL_DEG))
cells = EV.assign(ix=((EV.lon - xmin) / DOM_CELL_DEG).astype(int).clip(0, nx - 1),
                  iy=((ymax - EV.lat) / DOM_CELL_DEG).astype(int).clip(0, ny - 1))
by = cells.groupby(["iy", "ix", "driver_group"]).area_ha.sum().reset_index()
dom = by.sort_values("area_ha").drop_duplicates(["iy", "ix"], keep="last")
dom_arr = np.full((ny, nx), np.nan)
dom_arr[dom.iy.to_numpy(), dom.ix.to_numpy()] = dom.driver_group.map({g: i for i, g in enumerate(GROUPS)}).to_numpy()
fig, ax = plt.subplots(figsize=(11, 8))
ax.imshow(np.where(study_arr > 0, 1, np.nan), extent=EXTENT, cmap=ListedColormap(["#e5f5e0"]))
ax.imshow(dom_arr, extent=EXTENT, cmap=ListedColormap(GCOL), vmin=-0.5, vmax=len(GROUPS) - 0.5, interpolation="nearest")
basemap(ax, f"Dominant driver per {DOM_CELL_DEG}° cell")
ax.legend(handles=[Patch(color=c, label=g) for g, c in zip(GROUPS, GCOL)], loc="lower left", fontsize=8)
plt.tight_layout(); save(fig, "map_dominant_driver"); plt.show()

#### 16 · Disturbance by distance to peat edge, roads and rivers
Cumulative share of disturbance events within each distance (as in Nesha et al., Fig. 5). Peat-edge distance uses events in the peat classes only; road and river distances use all events.

In [ ]:
EVp = EV[EV.zone != "buffer"]
cur = pd.DataFrame({"peat edge (peat classes)": cum_share(EVp, "edge_bin", EDGE_BINS_KM)})
acc = pd.DataFrame({"road": cum_share(EV, "road_bin", ACCESS_BINS_KM),
                    "river": cum_share(EV, "river_bin", ACCESS_BINS_KM),
                    "road or river": cum_share(EV, "access_bin", ACCESS_BINS_KM)})
print("Cumulative % of events"); display(cur.T.round(1)); display(acc.T.round(1))
print("Cumulative % of disturbed area (pixel basis)")
display(pd.DataFrame({"peat edge (peat classes)": cum_share(D[D.zone != "buffer"], "edge_bin", EDGE_BINS_KM, "area_ha")}).T.round(1))
display(pd.DataFrame({k: cum_share(D, c, ACCESS_BINS_KM, "area_ha") for k, c in
                      [("road", "road_bin"), ("river", "river_bin"), ("road or river", "access_bin")]}).T.round(1))

fig, axes = plt.subplots(3, 1, figsize=(12, 16))
axes[0].plot(EDGE_BINS_KM, cur.iloc[:-1, 0], marker="o", color="#e34a33", label="Peat edge")
for col, c in zip(acc.columns, ["#7f0000", "#2b4bd8", "#1b9e77"]):
    axes[0].plot(ACCESS_BINS_KM, acc[col].iloc[:-1], marker="o", color=c, label=col.capitalize())
axes[0].set_title("All events")
for ax, (lab, col, edges, df) in zip(axes[1:], [("peat edge", "edge_bin", EDGE_BINS_KM, EVp),
                                                ("road or river", "access_bin", ACCESS_BINS_KM, EV)]):
    for g, c in zip(GROUPS, GCOL):
        e = df[df.driver_group == g]
        if len(e) >= 20:
            ax.plot(edges, cum_share(e, col, edges).iloc[:-1], marker=".", color=c, label=g)
    ax.set_title(f"By driver — distance to {lab}")
for ax in axes:
    ax.set_xlabel("Distance (km)"); ax.set_ylabel("Cumulative % of events"); ax.set_ylim(0, 101); ax.legend(fontsize=7)
plt.tight_layout(); save(fig, "distance_curves"); plt.show()

print("Key thresholds by driver (% of events)")
def thresholds(g):
    gp = g[g.zone != "buffer"]
    return {"≤1 km of peat edge": cum_share(gp, "edge_bin", EDGE_BINS_KM)["≤1 km"],
            "≤3 km of peat edge": cum_share(gp, "edge_bin", EDGE_BINS_KM)["≤3 km"],
            "≤5 km of road": cum_share(g, "road_bin", ACCESS_BINS_KM)["≤5 km"],
            "≤5 km of river": cum_share(g, "river_bin", ACCESS_BINS_KM)["≤5 km"],
            "≤10 km of road or river": cum_share(g, "access_bin", ACCESS_BINS_KM)["≤10 km"],
            "events": len(g)}
display(pd.DataFrame({grp: thresholds(EV[EV.driver_group == grp]) for grp in GROUPS}).T.round(1))

#### 17 · Exports for GIS and dashboards
Written to `outputs_peat_disturbance/gis/` and `outputs_peat_disturbance/dashboard/`.

**GeoTIFF** (`EXPORT_GEOTIFF`): `peat_disturbance_grid_<RUN_TAG>.tif`, EPSG:4326, one ~1 km pixel per grid cell (`GRID_DEG`), float32, NaN = no data. A `…_bands.csv` file lists the band meanings and the code tables.

**Dashboard tables** (`EXPORT_DASHBOARD_TABLES`), all plain CSV:
| file | one row per | coordinates / join keys |
|---|---|---|
| `events_points` | disturbance event | `lon`, `lat` (event centre) |
| `grid_cells_by_year` | ~1 km cell × year | `lon`, `lat` (cell centre), `cell_id` |
| `grid_cells_total` | ~1 km cell (whole period) | `lon`, `lat`, `cell_id` |
| `summary_monthly` | year × month × driver × class | — |
| `summary_by_area` | year × driver × class × protected area × concession × admin unit | names match the polygon layers' name fields (`PA_FIELD`, `CONC_FIELD`, `ADM_FIELD`) for joins |
| `study_area_by_area` | class × protected area × concession × admin unit | same join keys (denominators for % disturbed) |

In [ ]:
GIS_DIR = LOCAL_OUT / "gis"; GIS_DIR.mkdir(exist_ok=True)
DASH_DIR = LOCAL_OUT / "dashboard"; DASH_DIR.mkdir(exist_ok=True)

def bin_label(edges):
    return {i: lab for i, lab in enumerate([f"0–{edges[0]:g} km"] + [f"{a:g}–{b:g} km" for a, b in zip(edges[:-1], edges[1:])]
                                           + [f">{edges[-1]:g} km"])}
EDGE_LAB, ACC_LAB = bin_label(EDGE_BINS_KM), bin_label(ACCESS_BINS_KM)

# main driver per ~1 km cell (largest event area)
ev_in = EV[inside].assign(cell=(row_i[inside] * NCOLS + col_i[inside]).astype("int64"))
main_drv = (ev_in.groupby(["cell", "driver_group"]).area_ha.sum().reset_index()
            .sort_values("area_ha").drop_duplicates("cell", keep="last").set_index("cell").driver_group)
DRV_CODE = {g: i + 1 for i, g in enumerate(GROUPS)}

# ── GeoTIFF ─────────────────────────────────────────────────────────────────
if EXPORT_GEOTIFF:
    bands = {"study_ha": to_arr(CELLS.study_ha.where(CELLS.study_ha > 0)),
             "disturbed_ha_all": to_arr(CELLS.dist_ha_all.where(CELLS.dist_ha_all > 0)),
             "intensity_pct": to_arr(CELLS.intensity_pct.where(CELLS.dist_ha_all > 0)),
             "intensity_class": cls_arr,
             **{f"disturbed_ha_{y}": to_arr(CELLS[y].where(CELLS[y] > 0)) for y in YEARS},
             "events": to_arr(evg.events.astype(float)),
             "median_event_ha": to_arr(evg.median_ha),
             "main_driver": to_arr(main_drv.map(DRV_CODE).astype(float))}
    tif = GIS_DIR / f"peat_disturbance_grid_{RUN_TAG}.tif"
    stack_arr = np.stack([b.astype("float32") for b in bands.values()])
    try:
        import rasterio
        from rasterio.transform import from_origin
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio"], check=True)
        import rasterio
        from rasterio.transform import from_origin
    with rasterio.open(tif, "w", driver="GTiff", height=NROWS, width=NCOLS, count=len(bands), dtype="float32",
                       crs="EPSG:4326", transform=from_origin(xmin, ymax, GRID_DEG, GRID_DEG), nodata=np.nan,
                       compress="deflate") as dst:
        dst.write(stack_arr)
        for i, name in enumerate(bands, start=1):
            dst.set_band_description(i, name)
        dst.update_tags(run=RUN_TAG, period=f"{START_YEAR}-{END_YEAR}", latest_alert=LATEST_ALERT_DATE,
                        intensity_class="1 low, 2 medium, 3 high/hotspot",
                        main_driver="; ".join(f"{v} {k}" for k, v in DRV_CODE.items()))
    legend = pd.DataFrame({"band": range(1, len(bands) + 1), "name": list(bands)})
    legend["meaning"] = legend.name.map({
        "study_ha": "study area in cell (ha)", "disturbed_ha_all": "disturbed area, whole period (ha)",
        "intensity_pct": "disturbed / study area (%)", "intensity_class": f"1 low (<{lo} %), 2 medium ({lo}–{hi} %), 3 high (>{hi} %)",
        "events": "number of events (by event centre)", "median_event_ha": "median event size (ha)",
        "main_driver": "; ".join(f"{v} = {k}" for k, v in DRV_CODE.items())}).fillna("disturbed area in that year (ha)")
    legend.to_csv(GIS_DIR / f"peat_disturbance_grid_{RUN_TAG}_bands.csv", index=False)
    print(f"GeoTIFF: {tif}  ({len(bands)} bands, {NCOLS} × {NROWS} px)"); display(legend.set_index("band"))

# ── dashboard tables ────────────────────────────────────────────────────────
if EXPORT_DASHBOARD_TABLES:
    ev_out = pd.DataFrame({
        "event_id": range(1, len(EV) + 1), "lon": EV.lon.round(6), "lat": EV.lat.round(6),
        "date": EV.date.dt.date, "year": EV.year, "month": EV.month,
        "driver": EV.driver, "driver_group": EV.driver_group, "with_fire": EV.with_fire,
        "landcover_class": EV.zone, "area_ha": EV.area_ha.round(4), "size_class": EV.size_class.astype(str),
        "protected_area": EV.pa, "concession": EV.conc, "admin_unit": EV.adm,
        "dist_peat_edge": EV.edge_bin.map(EDGE_LAB).where(EV.zone != "buffer"),
        "dist_road": EV.road_bin.map(ACC_LAB), "dist_river": EV.river_bin.map(ACC_LAB),
        "dist_road_or_river": EV.access_bin.map(ACC_LAB), "below_mmu": EV.below_mmu})
    cells_y = (CELLS[YEARS + ["study_ha", "lon", "lat"]].reset_index(names="cell_id")
               .melt(id_vars=["cell_id", "lon", "lat", "study_ha"], value_vars=YEARS, var_name="year", value_name="disturbed_ha"))
    cells_y = cells_y[cells_y.disturbed_ha > 0]
    cells_y["intensity_pct"] = (cells_y.disturbed_ha / cells_y.study_ha.where(cells_y.study_ha > 0) * 100).clip(upper=100)
    cells_t = (CELLS[["lon", "lat", "study_ha", "dist_ha_all", "intensity_pct", "class"]]
               .join(evg[["events", "median_ha"]]).join(main_drv.rename("main_driver"))
               .reset_index(names="cell_id").rename(columns={"dist_ha_all": "disturbed_ha", "class": "intensity_class"}))
    cells_t = cells_t[cells_t.study_ha > 0]
    monthly = (D.groupby(["year", "month", "driver", "driver_group", "zone"]).area_ha.sum().reset_index()
               .rename(columns={"zone": "landcover_class", "area_ha": "disturbed_ha"}))
    by_area = (P.groupby(["year", "driver", "driver_group", "zone", "pa", "conc", "adm"]).area_ha.sum().reset_index()
               .rename(columns={"zone": "landcover_class", "pa": "protected_area", "conc": "concession",
                                "adm": "admin_unit", "area_ha": "disturbed_ha"}))
    study_area = (ZP.rename(columns={"zone": "landcover_class", "pa": "protected_area", "conc": "concession",
                                     "adm": "admin_unit", "area_ha": "study_ha"}))
    study_area = study_area[["landcover_class", "protected_area", "concession", "admin_unit", "study_ha"]]
    out = {"events_points": ev_out, "grid_cells_by_year": cells_y, "grid_cells_total": cells_t,
           "summary_monthly": monthly, "summary_by_area": by_area, "study_area_by_area": study_area}
    for name, df in out.items():
        df.to_csv(DASH_DIR / f"{name}_{RUN_TAG}.csv", index=False, float_format="%.4f")
    display(pd.DataFrame({"rows": {k: len(v) for k, v in out.items()},
                          "columns": {k: ", ".join(v.columns) for k, v in out.items()}}))
    print(f"Dashboard tables in {DASH_DIR}/ — names in protected_area / concession / admin_unit match the "
          f"'{PA_FIELD}', '{CONC_FIELD}' and '{ADM_FIELD}' fields of the polygon layers (outside = '— …').")

#### 18 · Notes
* **Disturbance definition.** A pixel counts if the WUR maps hold an alert dated `START_YEAR`–`END_YEAR`; the most recent alert and its class are kept. The last ~3 months before the latest alert are provisional, and the current year is partial (dashed in graphs).
* **No forest mask.** The study area is the land-cover classes themselves, so intensity is the disturbed share of each cell's study area.
* **Distances.** Peat-edge distance comes from `PEAT_DIST_ASSET`. Road and river distances are computed at 100 m, capped at 30 km, after masking the features to the selected country, so roads or rivers just across the border are ignored.
* **Protected areas, concessions, admin units** are painted by polygon name; where polygons overlap, the last one painted wins. The country mask also applies to them.
* **Events** are 8-connected pixels over the whole period and are split at processing-tile boundaries. Events smaller than `MMU_HA` are mostly fragments created by the class boundaries; they are flagged in `below_mmu`.